In [49]:
#1 The Data

In [377]:
import torch
from torch_geometric.datasets import MovieLens100K

In [378]:
dataset = MovieLens100K(root = "./data")

In [379]:
#data contains 1 graph
len(dataset)

1

In [380]:
#graph info
data = dataset[0]
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)


In [381]:
print(data.node_types)

['movie', 'user']


In [382]:
print(data.edge_types)

[('user', 'rates', 'movie'), ('movie', 'rated_by', 'user')]


In [383]:
#movie vectors represent movie genres (one movie can belong to several genres).
movie_x = data["movie"].x
print(movie_x[:5])

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


In [384]:
# Create Genre Relation

In [385]:
#since there as no edge relation in the graph, we need to reconstruct it from the movie embeddings.
#with torch nonzero define the exact place of each "movie-genre" correspondence in movie_x 
movie_ids, genre_ids = movie_x.nonzero(as_tuple=True)
#create a new edge type in the data
data["movie", "has_genre", "genre"].edge_index = torch.stack([movie_ids, genre_ids], dim=0)
data["genre"].genres = genre_ids
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ genres=[2891] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={ edge_index=[2, 2891] }
)


In [386]:
#2 GCN baseline

In [387]:
#2.1. Transform edge index for GCN
#Shift movie indices and unite them with user indices in one raw: 

In [388]:
edge_index = data["user", "rates", "movie"].edge_index
print(edge_index)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])


In [389]:
movie_index = edge_index[1]
print(movie_index)

tensor([   0,    1,    2,  ..., 1187, 1227, 1329])


In [390]:
number_users = data["user"].num_nodes
print(number_users)

943


In [391]:
transformed_movie_index = movie_index + number_users
print(transformed_movie_index)

tensor([ 943,  944,  945,  ..., 2130, 2170, 2272])


In [392]:
user_index = edge_index[0]
print(user_index)

tensor([  0,   0,   0,  ..., 942, 942, 942])


In [393]:
pos_edge_index = torch.stack([user_index, transformed_movie_index], dim=0)
print(pos_edge_index)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [ 943,  944,  945,  ..., 2130, 2170, 2272]])


In [394]:
#2.2. Sample negative examples

In [395]:
number_movies = data["movie"].num_nodes
print(number_movies)

1682


In [396]:
number_pos = data["user", "rates", "movie"].edge_index.shape[1]
print(number_pos)

80000


In [397]:
from torch_geometric.utils import negative_sampling
neg_edge_index = negative_sampling(edge_index, (number_users, number_movies), number_pos)
print(neg_edge_index)
print(neg_edge_index.shape)

tensor([[ 405,  462,  756,  ...,  268,  771,  744],
        [ 668, 1374, 1030,  ...,  679, 1003,  950]])
torch.Size([2, 80000])


In [398]:
neg_movie_index = neg_edge_index[1]
print(neg_movie_index)

tensor([ 668, 1374, 1030,  ...,  679, 1003,  950])


In [399]:
transform_neg_movie_index = neg_movie_index + number_users
print(transform_neg_movie_index)

tensor([1611, 2317, 1973,  ..., 1622, 1946, 1893])


In [400]:
neg_user_index = neg_edge_index[0]
print(neg_user_index)

tensor([405, 462, 756,  ..., 268, 771, 744])


In [401]:
neg_edge_index = torch.stack([neg_user_index, transform_neg_movie_index], dim=0)
print(neg_edge_index)

tensor([[ 405,  462,  756,  ...,  268,  771,  744],
        [1611, 2317, 1973,  ..., 1622, 1946, 1893]])


In [402]:
#2.3. Create training data

In [403]:
training_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
print(training_edge_index)
print(training_edge_index.shape)

tensor([[   0,    0,    0,  ...,  268,  771,  744],
        [ 943,  944,  945,  ..., 1622, 1946, 1893]])
torch.Size([2, 160000])


In [404]:
y_1 = torch.ones(pos_edge_index.shape[1])
print(y_1)
print(y_1.shape)
y_2 = torch.zeros(neg_edge_index.shape[1])
print(y_2)
print(y_2.shape)

tensor([1., 1., 1.,  ..., 1., 1., 1.])
torch.Size([80000])
tensor([0., 0., 0.,  ..., 0., 0., 0.])
torch.Size([80000])


In [405]:
y_train = torch.cat([y_1, y_2])
print(y_train.shape)

torch.Size([160000])


In [406]:
#2.4. Create GCN Recommender

In [407]:
feature_users = data["user"].x.shape[1]
print(feature_users)

24


In [408]:
feature_movies = data["movie"].x.shape[1]
print(feature_movies)

18


In [409]:
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GCNConv

In [410]:
class GCNModel(nn.Module):
    def __init__(self, init_dim_users,
                 init_dim_movies,
                 dim_unified,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.transform_users = nn.Linear(init_dim_users, dim_unified)
        self.transform_movies = nn.Linear(init_dim_movies, dim_unified)
        self.gcn1 = GCNConv(dim_unified, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, embed_dim)
        self.head = nn.Linear(embed_dim * 2, prediction_binary)
    def forward(self, users, movies, message_pass_adj, training_adj):
        #Your code goes here#
        u = self.transform_users(users)
        m = self.transform_movies(movies)
        x = torch.cat([u, m], dim=0)
        x = self.gcn1.forward(x, message_pass_adj)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.gcn2.forward(x, message_pass_adj)
        x = F.dropout(x, p=0.5, training=self.training)
        x = torch.cat([ x[training_adj[0]], x[training_adj[1]] ], dim=1)
        x = self.head(x)
        return x

In [411]:
#2.5. Train Recommender

In [412]:
model = GCNModel(init_dim_users=feature_users, init_dim_movies=feature_movies, dim_unified=32, hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [413]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model.forward(users=data["user"].x, movies=data["movie"].x, message_pass_adj=pos_edge_index, training_adj=training_edge_index)
    l = loss(pred.squeeze(1), y_train)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.6604015231132507, accuracy: 0.570687472820282
epoch 20, CEL: 0.5657562613487244, accuracy: 0.7249187231063843
epoch 40, CEL: 0.5412619113922119, accuracy: 0.732699990272522
epoch 60, CEL: 0.5292859077453613, accuracy: 0.7384999990463257
epoch 80, CEL: 0.5153871774673462, accuracy: 0.7436812520027161
epoch 100, CEL: 0.5148974061012268, accuracy: 0.7473124861717224
epoch 120, CEL: 0.5102670788764954, accuracy: 0.7518125176429749
epoch 140, CEL: 0.5074644684791565, accuracy: 0.753068745136261
epoch 160, CEL: 0.508330762386322, accuracy: 0.7512312531471252
epoch 180, CEL: 0.5037071704864502, accuracy: 0.7538062334060669
epoch 200, CEL: 0.505250871181488, accuracy: 0.7539812326431274


In [414]:
#2.6 Evaluate Recommender

In [415]:
#2.6.1 Transform edge index for GCN
#Shift movie indices and unite them with user indices in one raw:

In [416]:
edge_label_index = data["user", "rates", "movie"].edge_label_index
print(edge_label_index)

tensor([[  0,   0,   0,  ..., 458, 459, 461],
        [  5,   9,  11,  ..., 933,   9, 681]])


In [417]:
movie_test_index = edge_label_index[1]
print(movie_test_index)

tensor([  5,   9,  11,  ..., 933,   9, 681])


In [418]:
print(number_users)

943


In [419]:
transform_movie_test_index = movie_test_index + number_users
print(transform_movie_test_index)

tensor([ 948,  952,  954,  ..., 1876,  952, 1624])


In [420]:
user_test_index = (data["user", "rates", "movie"].edge_label_index)[0]
print(user_test_index)

tensor([  0,   0,   0,  ..., 458, 459, 461])


In [421]:
pos_edge_test_index = torch.stack([user_test_index, transform_movie_test_index], dim=0)
print(pos_edge_test_idx)

tensor([[   0,    0,    0,  ...,  458,  459,  461],
        [ 948,  952,  954,  ..., 1876,  952, 1624]])


In [422]:
#2.6.2 Sample negative examples

In [423]:
print(number_users)

943


In [424]:
print(number_movies)

1682


In [425]:
number_pos_test = data["user", "rates", "movie"].edge_label_index.shape[1]
print(number_pos_test)

20000


In [426]:
from torch_geometric.utils import negative_sampling
neg_edge_test_index = negative_sampling(edge_label_index, (number_users, number_movies), number_pos_test)
print(neg_edge_test_index)
print(neg_edge_test_index.shape)

tensor([[ 112,  183,  564,  ...,  565,  809,  233],
        [  56, 1040,  569,  ...,  451,  902, 1469]])
torch.Size([2, 20000])


In [427]:
transform_movie_neg_edge_test_index = neg_edge_test_index[1] + number_users
print(transform_movie_neg_edge_test_index)

tensor([ 999, 1983, 1512,  ..., 1394, 1845, 2412])


In [428]:
user_neg_edge_test_index = neg_edge_test_index[0]
print(user_neg_edge_test_index)

tensor([112, 183, 564,  ..., 565, 809, 233])


In [431]:
neg_edge_test_index = torch.stack([user_neg_edge_test_index, transform_movie_neg_edge_test_index], dim=0)
print(neg_edge_test_index)

tensor([[ 112,  183,  564,  ...,  565,  809,  233],
        [ 999, 1983, 1512,  ..., 1394, 1845, 2412]])


In [239]:
#2.6.2 Create test data

In [432]:
test_edge_index = torch.cat([pos_edge_test_index, neg_edge_test_index], dim=1)
print(test_edge_index)
print(test_edge_index.shape)

tensor([[   0,    0,    0,  ...,  565,  809,  233],
        [ 948,  952,  954,  ..., 1394, 1845, 2412]])
torch.Size([2, 40000])


In [433]:
y_test_1 = torch.ones(pos_edge_test_idx.shape[1])
print(y_1)
print(y_1.shape)
y_test_2 = torch.zeros(neg_edge_test_idx.shape[1])
print(y_2)
print(y_2.shape)

tensor([1., 1., 1.,  ..., 1., 1., 1.])
torch.Size([80000])
tensor([0., 0., 0.,  ..., 0., 0., 0.])
torch.Size([80000])


In [434]:
y_test = torch.cat([y_test_1, y_test_2])
print(y_test.shape)

torch.Size([40000])


In [343]:
#2.6.3. Launch evaluation

In [435]:
model.eval()
with torch.no_grad():
    pred_test = model.forward(users=data["user"].x, movies=data["movie"].x, message_pass_adj=pos_edge_test_index, training_adj=test_edge_index)
    l_test = loss(pred_test.squeeze(1), y_test)
    predictions_test = (pred_test.sigmoid() >= 0.5).float()
    accuracy_test = (predictions_test.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test}, test accuracy: {accuracy_test}")

test CEL: 0.6433544158935547, test accuracy: 0.5985249876976013


In [436]:
#3. R-GCN

In [437]:
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ genres=[2891] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={ edge_index=[2, 2891] }
)


In [438]:
#3.1. Set arguments for R-GCN layer

In [267]:
movie = data["movie"].x
print(movie)

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [269]:
user = data["user"].x

In [281]:
pos_edge_index_relational = data["user", "rates", "movie"].edge_index
print(pos_edge_index_relational)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])


In [547]:
relations = data["user", "rates", "movie"].rating
print(relations)

tensor([5, 3, 4,  ..., 3, 3, 3])


In [277]:
#3.2. Create relational training data

In [440]:
print(training_edge_index)

tensor([[   0,    0,    0,  ...,  268,  771,  744],
        [ 943,  944,  945,  ..., 1622, 1946, 1893]])


In [448]:
transform_back = training_edge_index[1] - number_users
print(transform_back)

tensor([   0,    1,    2,  ...,  679, 1003,  950])


In [553]:
training_users = training_edge_index[0]
print(training_users)

tensor([  0,   0,   0,  ..., 268, 771, 744])


In [554]:
training_edge_index_relational = torch.stack([training_users, transform_back], dim=0)
print(training_edge_index_relational)

tensor([[   0,    0,    0,  ...,  268,  771,  744],
        [   0,    1,    2,  ...,  679, 1003,  950]])


In [555]:
y_train_relational = y_train
print(y_train_relational)
print(y_train.shape)

tensor([1., 1., 1.,  ..., 0., 0., 0.])
torch.Size([160000])


In [453]:
#3.3. Create RGCN Recommender

In [454]:
from torch_geometric.nn import RGCNConv

In [455]:
feature_users = data["user"].x.shape[1]
print(feature_users)

24


In [456]:
feature_movies = data["movie"].x.shape[1]
print(feature_movies)

18


In [538]:
class RGCNModel(nn.Module):
    def __init__(self, dim_users,
                 dim_movies,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.rgcn1 = RGCNConv((dim_users, dim_movies), hidden_dim, num_relations=5+1)
        self.rgcn2 = RGCNConv((dim_users, hidden_dim), embed_dim, num_relations=5+1)
        self.head = nn.Linear(embed_dim + feature_users, prediction_binary)
    def forward(self, user, movie, message_pass_adj_relational, relation_types, training_adj_relational):
        #Your code goes here#
        m_emb = self.rgcn1.forward((user, movie), message_pass_adj_relational, relation_types)
        m_emb = F.relu(m_emb)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        m_emb = self.rgcn2.forward((user, m_emb), message_pass_adj_relational, relation_types)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        x = torch.cat([ user[training_adj_relational[0]], m_emb[training_adj_relational[1]] ], dim=1)
        x = self.head(x)
        return x

In [539]:
#3.4. Train RGCN

In [540]:
model_rgcn = RGCNModel(dim_users=feature_users, dim_movies=feature_movies, hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model_rgcn.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [542]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model_rgcn.forward(user=user, movie=movie, message_pass_adj_relational=pos_edge_index_relational,
                              relation_types=relations, training_adj_relational=training_edge_index_relational)
    l = loss(pred.squeeze(1), y_train_relational)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.7032959461212158, accuracy: 0.5386187434196472
epoch 20, CEL: 0.5450448989868164, accuracy: 0.7294062376022339
epoch 40, CEL: 0.5274904370307922, accuracy: 0.7380874752998352
epoch 60, CEL: 0.5203932523727417, accuracy: 0.7450500130653381
epoch 80, CEL: 0.5168943405151367, accuracy: 0.7462000250816345
epoch 100, CEL: 0.5148281455039978, accuracy: 0.7482374906539917
epoch 120, CEL: 0.5111835598945618, accuracy: 0.7499625086784363
epoch 140, CEL: 0.5085496306419373, accuracy: 0.7501999735832214
epoch 160, CEL: 0.5085753798484802, accuracy: 0.7516312599182129
epoch 180, CEL: 0.5077105164527893, accuracy: 0.7522125244140625
epoch 200, CEL: 0.5097949504852295, accuracy: 0.7515937685966492


In [543]:
#3.5. Evaluate RGCN Recommender

In [570]:
test_pos_edge_index_relational = data["user", "rates", "movie"].edge_label_index
print(test_pos_edge_index_relational)
print(test_pos_edge_index_relational.shape)

tensor([[  0,   0,   0,  ..., 458, 459, 461],
        [  5,   9,  11,  ..., 933,   9, 681]])
torch.Size([2, 20000])


In [568]:
test_relations = data["user", "rates", "movie"].edge_label.long()
print(test_relations)

tensor([5, 3, 5,  ..., 3, 3, 5])


In [575]:
print(test_edge_index)
print(test_edge_index.shape)

tensor([[   0,    0,    0,  ...,  565,  809,  233],
        [ 948,  952,  954,  ..., 1394, 1845, 2412]])
torch.Size([2, 40000])


In [576]:
test_transform_back = test_edge_index[1] - number_users
print(test_transform_back)

tensor([   5,    9,   11,  ...,  451,  902, 1469])


In [578]:
test_users = test_edge_index[0]
print(test_users)

tensor([  0,   0,   0,  ..., 565, 809, 233])


In [579]:
test_edge_index_relational = torch.stack([test_users, test_transform_back], dim=0)
print(test_edge_index_relational)
print(test_edge_index_relational.shape)

tensor([[   0,    0,    0,  ...,  565,  809,  233],
        [   5,    9,   11,  ...,  451,  902, 1469]])
torch.Size([2, 40000])


In [580]:
y_test_relational = y_test
print(y_test_relational)
print(y_test.shape)

tensor([1., 1., 1.,  ..., 0., 0., 0.])
torch.Size([40000])


In [ ]:
#3.6. Launch Evaluation

In [581]:
model_rgcn.eval()
with torch.no_grad():
    pred_test_relational = model_rgcn.forward(user=user, movie=movie, message_pass_adj_relational=test_pos_edge_index_relational,
                              relation_types=test_relations, training_adj_relational=test_edge_index_relational)
    l_test_relational = loss(pred_test_relational.squeeze(1), y_test_relational)
    predictions_test_relational = (pred_test_relational.sigmoid() >= 0.5).float()
    accuracy_test_relational = (predictions_test_relational.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test_relational}, test accuracy: {accuracy_test_relational}")

test CEL: 0.6102417707443237, test accuracy: 0.6865249872207642
